In [ ]:
!git clone https://github.com/Tencent/Hunyuan3D-2
%cd Hunyuan3D-2

!pip install -r requirements.txt
!pip install -e .
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install rembg
%cd /kaggle/working/Hunyuan3D-2
!cd /kaggle/working/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer && python3 setup.py build_ext --inplace

!cd /kaggle/working/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer && python3 setup.py build_ext --inplace

In [ ]:
!pip install fastapi uvicorn nest_asyncio pyngrok python-multipart rembg

In [ ]:
import sys, os, torch, uvicorn, nest_asyncio, asyncio, io, uuid, gc, time, threading, logging
from contextlib import redirect_stdout, redirect_stderr
import numpy as np
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
from PIL import Image
from rembg import remove, new_session

sys.path.append('/kaggle/working/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer')
sys.path.append('/kaggle/working/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer')

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.texgen import Hunyuan3DPaintPipeline

# Best-quality rembg model for cleaner masks
rembg_session = new_session("birefnet-general")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("hunyuan3d")

print("Loading Shape Pipe to GPU 0...")
shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
shape_pipe.to('cuda:0', dtype=torch.float16)
if hasattr(shape_pipe, "set_progress_bar_config"):
    shape_pipe.set_progress_bar_config(disable=True)

print("Loading Texture Pipe to GPU 1...")
with torch.cuda.device(1):
    tex_pipe = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2', subfolder='hunyuan3d-paint-v2-0')
    if hasattr(tex_pipe, 'models'):
        for name, m in tex_pipe.models.items():
            if hasattr(m, 'to'):
                m.to('cuda:1')
    if hasattr(tex_pipe, 'device'):
        tex_pipe.device = torch.device('cuda:1')
    if hasattr(tex_pipe, "set_progress_bar_config"):
        tex_pipe.set_progress_bar_config(disable=True)


def preprocess(img_bytes: bytes, size: int = 512) -> Image.Image:
    """Remove background, center subject, pad to square, resize to size x size."""
    img = Image.open(io.BytesIO(img_bytes)).convert("RGBA")

    img = remove(img, session=rembg_session)

    bbox = img.getbbox()
    if bbox:
        img = img.crop(bbox)

    w, h = img.size
    side = max(w, h)
    square = Image.new("RGBA", (side, side), (0, 0, 0, 0))
    square.paste(img, ((side - w) // 2, (side - h) // 2))

    img = square.resize((size, size), Image.LANCZOS)
    return img


app = FastAPI()

tasks = {}
tasks_lock = threading.Lock()


def _update_task(job_id: str, payload: dict) -> None:
    with tasks_lock:
        current = tasks.get(job_id, {})
        current.update(payload)
        tasks[job_id] = current


def _get_task(job_id: str):
    with tasks_lock:
        return tasks.get(job_id)


def run_gen_sync(job_id: str, img_bytes: bytes) -> None:
    _update_task(job_id, {"status": "processing", "started_at": time.time()})
    logger.info("job %s start", job_id)
    log_buffer = io.StringIO()
    try:
        with redirect_stdout(log_buffer), redirect_stderr(log_buffer), torch.inference_mode():
            img = preprocess(img_bytes, size=512)
            logger.info("job %s preprocess done", job_id)

            mesh = shape_pipe(
                image=img,
                num_inference_steps=100,
                guidance_scale=7.5,
                mc_resolution=256,
            )[0]
            logger.info("job %s mesh done", job_id)

            with torch.cuda.device(1):
                textured_mesh = tex_pipe(mesh=mesh, image=img)
            logger.info("job %s texture done", job_id)

            save_path = f"/kaggle/working/{job_id}.glb"
            textured_mesh.export(save_path)
            _update_task(job_id, {"status": "completed", "file": save_path, "finished_at": time.time()})
            logger.info("job %s saved %s", job_id, save_path)
    except Exception as e:
        logger.exception("job %s failed", job_id)
        log_tail = log_buffer.getvalue()[-2000:] if log_buffer.getvalue() else None
        payload = {"status": "failed", "error": str(e), "finished_at": time.time()}
        if log_tail:
            payload["log_tail"] = log_tail
        _update_task(job_id, payload)
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


@app.post("/generate")
async def start_gen(file: UploadFile = File(...), job_id: str = None):
    job_id = job_id or str(uuid.uuid4())
    img_bytes = await file.read()
    if not img_bytes:
        raise HTTPException(status_code=400, detail="Empty file")
    _update_task(job_id, {"status": "processing", "queued_at": time.time()})
    logger.info("job %s received bytes=%d", job_id, len(img_bytes))
    asyncio.create_task(asyncio.to_thread(run_gen_sync, job_id, img_bytes))
    return {"job_id": job_id, "status": "processing"}


@app.get("/status/{job_id}")
async def get_status(job_id: str):
    task = _get_task(job_id)
    if not task:
        return {"status": "not_found"}
    return task


@app.get("/download/{job_id}")
async def download(job_id: str):
    task = _get_task(job_id)
    if task and task.get("status") == "completed":
        return FileResponse(task["file"], media_type="model/gltf-binary", filename=f"{job_id}.glb")
    return {"error": "Not ready"}


nest_asyncio.apply()
ngrok.kill()
token = "3CfBzUtwUpneeSusnOZCk4VQ1xm_2PU7WRR7TgxBAm4iDo5gB"
ngrok.set_auth_token(token)
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
loop = asyncio.get_event_loop()
loop.create_task(server.serve())
public_url = ngrok.connect(8000)
logger.info("API is live at: %s", public_url.public_url)
print(f"API is live at: {public_url.public_url}")

In [ ]:
!nvidia-smi